# EV06: limpieza, anÃ¡lisis y estructuraciÃ³n de datos de Instacart con Pandas

**Pregunta orientadora:** Â¿CÃ³mo preparar y estructurar los datos de Instacart para obtener informaciÃ³n confiable sobre los hÃ¡bitos de compra?

Este cuaderno desarrolla, en orden, las lecciones 2, 3 y 4 del material [Preparar y estructurar datos con Python](https://educated-show-144.notion.site/Preparar-y-estructurar-datos-con-Python-40f4671e02a180dd9642deb111c14df2): los ejemplos de cada lecciÃ³n, su laboratorio guiado y los siete ejercicios de su reto. Cada lecciÃ³n usa las variables de la anterior, por lo que debe ejecutarse de arriba abajo (*Kernel â†’ Restart & Run All*).

- Datos originales (sin modificar): `data/raw/`, separados por punto y coma.
- Archivos generados: `data/processed/`.

## PreparaciÃ³n: rutas y carga de las cinco tablas

La carga repite el punto de partida de la LecciÃ³n 1 (`sep=';'`). El cuaderno funciona tanto si se abre desde la raÃ­z del proyecto como desde `notebooks/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd() if (Path.cwd() / 'data').is_dir() else Path.cwd().parent
RUTA = RAIZ / 'data' / 'raw'          # datos crudos
SALIDA = RAIZ / 'data' / 'processed'  # archivos generados
SALIDA.mkdir(parents=True, exist_ok=True)

orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

print('pandas', pd.__version__)
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

pandas 3.1.1
orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


---
# LecciÃ³n 2: Limpieza y estandarizaciÃ³n de datos crudos

Se resuelven los problemas detectados en la LecciÃ³n 1: duplicados, tipos incorrectos, nulos, texto sin estandarizar y duplicados implÃ­citos. Para cada problema se decide **eliminar, corregir o conservar** segÃºn lo que significa el dato. Orden: duplicados â†’ tipos â†’ nulos â†’ texto â†’ duplicados implÃ­citos â†’ verificaciÃ³n.

## 2.1 Duplicados exactos: `duplicated` y `drop_duplicates`

In [2]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


In [3]:
print('Franjas de los duplicados:', duplicados[['order_dow', 'order_hour_of_day']].drop_duplicates().values.tolist())
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

Franjas de los duplicados: [[3, 2]]


(478952, 6) True


Las 30 filas son 15 pedidos y sus copias, todas el miÃ©rcoles (`order_dow = 3`) a las 2 a. m. Que coincidan en la misma franja apunta a un fallo del sistema y no a pedidos reales, asÃ­ que se **eliminan**.

## 2.2 Tipos de dato con `astype`

In [4]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('DespuÃ©s:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB


DespuÃ©s: 7.2 MB


`Int16` (con mayÃºscula) es un entero *nullable*: admite los nulos de `days_since_prior_order` y `add_to_cart_order` sin convertirlos en decimales.

## 2.3 Nulos: comprobar la causa antes de decidir

In [5]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


In [6]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


| Columna | Causa | DecisiÃ³n |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente: no existe pedido anterior | Conservar el nulo (rellenar con 0 inventarÃ­a una espera de 0 dÃ­as) |
| `product_name` | Productos del pasillo 100 / departamento 21 (`missing`), sin datos de catÃ¡logo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2, ejercicio 3 | Conservar el nulo |

## 2.4 Texto con los mÃ©todos `.str`

In [7]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16


0 0


## 2.5 Duplicados implÃ­citos

In [8]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141


268 de 280


Tras estandarizar aparecen nombres repetidos con `product_id` distintos y casi todos se usan en pedidos. DecisiÃ³n: **conservar y documentar**; borrarlos dejarÃ­a pedidos apuntando a productos inexistentes. Los anÃ¡lisis agrupan por `product_id`, no por nombre.

## 2.6 Limpieza reutilizable con funciones y verificaciÃ³n

In [9]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


In [10]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Limpieza verificada.')

Limpieza verificada.


## Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Se construye un **pipeline** que parte de los archivos crudos, se ejecuta con una sola instrucciÃ³n y deja un **informe** de lo que cambiÃ³ en cada paso. Trabaja sobre copias nuevas, sin tocar las tablas ya limpias.

### Paso 1: recargar los datos crudos

In [11]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


### Paso 2: una funciÃ³n que anota el estado de la tabla

In [12]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

### Paso 3: un paso de limpieza por funciÃ³n

In [13]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

### Paso 4: encadenar los pasos con `pipe`

In [14]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

### Paso 5: el informe de cambios

In [15]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**InterpretaciÃ³n del informe:**

- Quitar duplicados eliminÃ³ 15 filas y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- Corregir tipos no cambiÃ³ filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- Los 836 nulos de `order_prods` se conservan a propÃ³sito.

### Paso 6: validar el resultado

In [16]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'ValidaciÃ³n superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

ValidaciÃ³n superada
True True


### Paso 7: guardar las tablas limpias y el informe

Se guardan en `data/processed/`. Parquet conserva los tipos (`int8`, `Int16`...) que CSV perderÃ­a.

In [17]:
informe.to_csv(SALIDA / 'cleaning_report.csv', index=False)
orders_limpio.to_parquet(SALIDA / 'orders_clean.parquet', index=False)
products_limpio.to_parquet(SALIDA / 'products_clean.parquet', index=False)
op_limpio.to_parquet(SALIDA / 'order_products_clean.parquet', index=False)
for archivo in ['cleaning_report.csv', 'orders_clean.parquet', 'products_clean.parquet', 'order_products_clean.parquet']:
    print(archivo, round((SALIDA / archivo).stat().st_size / 1e6, 2), 'MB')

cleaning_report.csv 0.0 MB
orders_clean.parquet 5.39 MB
products_clean.parquet 1.45 MB
order_products_clean.parquet 32.51 MB


### Conclusiones clave

- Un pipeline es una cadena de funciones pequeÃ±as que reciben y devuelven un DataFrame, unidas con `pipe`.
- Intercalar una funciÃ³n de registro produce un informe de cambios sin esfuerzo extra.
- El informe responde con cifras a la pregunta "Â¿quÃ© le hiciste a los datos?".
- Validar con `assert` al final convierte las reglas de calidad en una prueba que se ejecuta cada vez.

## Reto 2

### Ejercicio 1. Duplicados en `order_prods` y valores de `reordered`

In [18]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]


**Respuesta:** `order_prods` no tiene filas duplicadas exactas ni pares `order_id` + `product_id` repetidos, y `reordered` solo contiene 0 y 1. No necesita limpieza de duplicados.

### Ejercicio 2. Nombres duplicados o con espacios en `aisles` y `departments`

In [19]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    print(nombre,
          '| duplicados:', df[col].duplicated().sum(),
          '| con espacios al inicio o final:', (df[col] != df[col].str.strip()).sum(),
          '| con espacios dobles:', df[col].str.contains('  ', regex=False).sum())

aisles | duplicados: 0 | con espacios al inicio o final: 0 | con espacios dobles: 0
departments | duplicados: 0 | con espacios al inicio o final: 0 | con espacios dobles: 0


**Respuesta:** Ni `aisles` (134 pasillos) ni `departments` (21 departamentos) tienen nombres duplicados, espacios al inicio o al final ni espacios dobles. No necesitan limpieza de texto; el Ãºnico detalle es el valor centinela `missing`.

### Ejercicio 3. Los 836 nulos de `add_to_cart_order`

In [20]:
pedidos_con_nulo = order_prods.loc[order_prods['add_to_cart_order'].isna(), 'order_id'].unique()
tamano_pedido = order_prods.groupby('order_id').size()
print('Pedidos con posiciÃ³n nula:', len(pedidos_con_nulo))
print('TamaÃ±o mÃ­nimo de esos pedidos:', tamano_pedido.loc[pedidos_con_nulo].min())
print('PosiciÃ³n mÃ¡xima registrada en todo el dataset:', order_prods['add_to_cart_order'].max())
nulos_por_pedido = order_prods.loc[order_prods['add_to_cart_order'].isna()].groupby('order_id').size()
print('Â¿Nulos = productos que pasan de la posiciÃ³n 64?',
      (tamano_pedido.loc[nulos_por_pedido.index] - 64 == nulos_por_pedido).all())

Pedidos con posiciÃ³n nula: 70
TamaÃ±o mÃ­nimo de esos pedidos: 65
PosiciÃ³n mÃ¡xima registrada en todo el dataset: 64
Â¿Nulos = productos que pasan de la posiciÃ³n 64? True


**Respuesta:** Los 836 nulos ocurren en 70 pedidos y cada uno tiene como mÃ­nimo 65 productos, mientras que la posiciÃ³n mÃ¡xima registrada es 64. Se comprobÃ³ que en cada pedido los nulos son exactamente los productos que pasan de la posiciÃ³n 64: el sistema solo registra las primeras 64 posiciones del carrito. Por eso se conservan como nulos; no es un error del pedido.

### Ejercicio 4. Nombres que cambiaron con la estandarizaciÃ³n

In [21]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
con_nombre = products_crudo['product_name'].notna()
cambiaron = (products_crudo.loc[con_nombre, 'product_name'] != products.loc[con_nombre, 'product_name']).sum()
print('Nombres no nulos que cambiaron:', cambiaron, 'de', con_nombre.sum())

Nombres no nulos que cambiaron: 6321 de 48436


**Respuesta:** 6 321 de los 48 436 nombres no nulos (13 %) cambiaron con la estandarizaciÃ³n: espacios dobles o invisibles y uso inconsistente de mayÃºsculas.

### Ejercicio 5. DÃ­a de la semana como categorÃ­a ordenada

In [22]:
dias = ['domingo', 'lunes', 'martes', 'miÃ©rcoles', 'jueves', 'viernes', 'sÃ¡bado']
dia_texto = orders['order_dow'].map(dict(enumerate(dias)))
dia_cat = pd.Series(pd.Categorical(dia_texto, categories=dias, ordered=True), index=orders.index)

print(dia_cat.cat.categories.tolist(), dia_cat.cat.ordered)
print('Memoria texto:', round(dia_texto.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Memoria categorÃ­a:', round(dia_cat.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Ordenada permite comparar:', (dia_cat > 'viernes').sum(), 'pedidos en sÃ¡bado')

['domingo', 'lunes', 'martes', 'miÃ©rcoles', 'jueves', 'viernes', 'sÃ¡bado'] True
Memoria texto: 7.38 MB
Memoria categorÃ­a: 0.48 MB
Ordenada permite comparar: 62649 pedidos en sÃ¡bado


**Respuesta:** La categorÃ­a ordenada (domingo < lunes < ... < sÃ¡bado) ocupa 0,48 MB frente a 7,38 MB del texto, unas 15 veces menos, porque guarda cada nombre una sola vez y un cÃ³digo por fila. AdemÃ¡s, permite comparar y ordenar los dÃ­as en orden de calendario y no alfabÃ©tico.

### Ejercicio 6. ArtÃ­culos de productos `'Unknown'`

In [23]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]
print('Filas de order_prods:', len(filas_unknown),
      f'({len(filas_unknown) / len(order_prods):.2%} del total)')
print('Pedidos en que aparecen:', filas_unknown['order_id'].nunique())

Filas de order_prods: 9547 (0.21% del total)
Pedidos en que aparecen: 8254


**Respuesta:** Los productos `Unknown` aparecen en 9 547 filas de `order_prods` (0,21 %), repartidas en 8 254 pedidos. No los eliminarÃ­a: son ventas reales, y borrarlos reducirÃ­a artificialmente el tamaÃ±o de esos pedidos y alterarÃ­a las tasas de recompra. Se conservan y se excluyen solo en los anÃ¡lisis por nombre de producto.

### Ejercicio 7. `'missing'` como nulo real en una copia de `departments`

In [24]:
departments_na = departments.copy()
departments_na['department'] = departments_na['department'].replace('missing', pd.NA)
print(departments_na.tail(2))
print('Nulos en la copia:', departments_na['department'].isna().sum(), '| en el original:', departments['department'].isna().sum())

    department_id department
19             20       deli
20             21        NaN
Nulos en la copia: 1 | en el original: 0


**Respuesta:** No conviene hacerlo en la tabla original porque `groupby` descarta los nulos por defecto (`dropna=True`): los productos del departamento 21 desaparecerÃ­an en silencio de los totales, mientras que con `'missing'` siguen visibles como un grupo propio.